# ツール関数と MCP サーバーの利用方法

このノートブックでは、ファンクションコールで外部の API サービスを呼び出す AI エージェントの作成方法を学びます。

ツール関数を独自に実装する方法と、既存の MCP サーバーを利用する方法をそれぞれ説明します。

## 事前準備

**[TFM-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。ここでは、特に、MCP ToolSet で必要なパッケージを追加しています。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk[mcp]==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[TFM-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[TMF-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[TMF-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[TMF-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [84]:
import os, requests
from typing import Dict, Any
from IPython.display import HTML, Markdown, display
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.adk.planners import BuiltInPlanner
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.colab import userdata
from google.genai.types import GenerateContentConfig, ThinkingConfig, ThinkingLevel

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[TMF-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        events = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            events.append(event)
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result), events

## ツール関数を独自に実装する例

**[TMF-07]**

Open-Meteo API を用いて、指定した都道府県の天気予報データを取得する関数 `get_weather_forecast` を定義します。

In [73]:
async def get_weather_forecast(prefecture_name: str) -> Dict[str, Any]:
    """
    指定された都道府県の今日から始まる7日間（168時間）の天気予報を取得します。

    Args:
        prefecture_name (str): 都道府県名
        - '東京都', '京都府', '鹿児島県' のみが利用可能

    Returns:
        Dict[str, Any]: Open-Meteo APIからの JSON レスポンスデータ

    Raises:
        ValueError: サポートされていない都道府県名が指定された場合
        requests.RequestException: API リクエストに失敗した場合

    JSON レスポンスデータのスキーマ:
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "OpenMeteoForecastResponse",
  "type": "object",
  "required": [
    "latitude",
    "longitude",
    "generationtime_ms",
    "utc_offset_seconds",
    "timezone",
    "timezone_abbreviation",
    "elevation",
    "hourly_units",
    "hourly"
  ],
  "properties": {
    "latitude": {
      "type": "number",
      "description": "APIが算出した対象地点の緯度"
    },
    "longitude": {
      "type": "number",
      "description": "APIが算出した対象地点の経度"
    },
    "generationtime_ms": {
      "type": "number",
      "description": "サーバー側でのレスポンス生成時間（ミリ秒）"
    },
    "utc_offset_seconds": {
      "type": "integer",
      "description": "UTC（協定世界時）からの時差（秒単位）。日本（JST）の場合は 32400"
    },
    "timezone": {
      "type": "string",
      "description": "対象地域のタイムゾーン名（例: 'Asia/Tokyo'）"
    },
    "timezone_abbreviation": {
      "type": "string",
      "description": "タイムゾーンの略称（例: 'JST' や 'GMT+9'）"
    },
    "elevation": {
      "type": "number",
      "description": "対象地点の標高（メートル）"
    },
    "hourly_units": {
      "type": "object",
      "description": "hourly オブジェクトに含まれる各データの表示単位",
      "required": [
        "time",
        "temperature_2m",
        "precipitation_probability",
        "rain",
        "weather_code"
      ],
      "properties": {
        "time": { "type": "string", "enum": ["iso8601"] },
        "temperature_2m": { "type": "string", "enum": ["°C", "°F"] },
        "precipitation_probability": { "type": "string", "enum": ["%"] },
        "rain": { "type": "string", "enum": ["mm", "inch"] },
        "weather_code": { "type": "string", "enum": ["wmo code"] }
      }
    },
    "hourly": {
      "type": "object",
      "description": "1時間ごとの時系列予報データ。配列要素に各時間のデータが紐付きます（通常168要素 = 7日分）。",
      "required": [
        "time",
        "temperature_2m",
        "precipitation_probability",
        "rain",
        "weather_code"
      ],
      "properties": {
        "time": {
          "type": "array",
          "items": { "type": "string", "format": "date-time" },
          "description": "ISO8601形式のDateTime文字列配列（例: '2026-09-02T00:00'）"
        },
        "temperature_2m": {
          "type": "array",
          "items": { "type": "number" },
          "description": "地上2mの気温の配列"
        },
        "precipitation_probability": {
          "type": "array",
          "items": { "type": "integer", "minimum": 0, "maximum": 100 },
          "description": "降水確率（0〜100%）の配列"
        },
        "rain": {
          "type": "array",
          "items": { "type": "number", "minimum": 0 },
          "description": "1時間あたりの雨量（mm）の配列"
        },
        "weather_code": {
          "type": "array",
          "items": { "type": "integer" },
          "description": "WMO天候コード（0:快晴, 1〜3:晴れ〜曇り, 51〜67:雨, 95〜99:雷雨 など）の配列"
        }
      }
    }
  }
}
    """

    prefecture_coordinates = {
        '東京都': {'lat': 35.6895, 'lon': 139.6917},
        '京都府': {'lat': 35.0211, 'lon': 135.7556},
        '鹿児島県': {'lat': 31.5602, 'lon': 130.5581},
    }
    if prefecture_name not in prefecture_coordinates:
        raise ValueError(f'{prefecture_name} はサポートされていません。')
    coords = prefecture_coordinates[prefecture_name]

    # Open-Meteo API エンドポイント
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': coords['lat'],
        'longitude': coords['lon'],
        'hourly': 'temperature_2m,precipitation_probability,rain,weather_code',
        'timezone': 'auto'
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()

**[TMF-08]**

関数 `get_weather_forecast` をツール関数に持つ LlmAgent オブジェクトと AdkApp オブジェクトを作成します。

In [85]:
instruction = '''
あなたは気象予報データに基づいて、ユーザーの質問に回答するエージェントです。
- get_weather_forecast で気象予報データを取得します。
- 取得したデータに基づいて、客観的な情報を提供してください。
- 気象情報については取得データに含まれる具体的な日付を含めてください。
'''

weather_forecast_agent = LlmAgent(
    name='weather_forecast_agent',
    model='gemini-3.8-flash',
    description='気象予報データに基づいて質問に回答するエージェント',
    instruction=instruction,
    tools=[get_weather_forecast],
    planner=BuiltInPlanner(
        thinking_config=ThinkingConfig(
            thinking_level=ThinkingLevel.LOW,
            include_thoughts=False,
        ),
    ),
)

weather_forecast_app = AdkApp(
    agent=weather_forecast_agent,
    app_name='weather_forecast_app',
)

**[TMF-09]**

作成したAIエージェントと会話します。

In [92]:
chat_client = ChatClient(weather_forecast_app)

query = '''
明日から3日間の京都旅行です。雨具の用意は必要ですか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

京都府の天気予報データ（2026年9月4日〜9月10日）に基づき、**明日（2026年9月5日）から3日間**の予報をお伝えします。

結論として、**雨具の用意が必要**です。特に2日目（9月6日）と3日目（9月7日）は雨が予想されます。

---

### 日ごとの詳細予報

- **2026年9月5日（土・1日目）**
  - **天候**: 曇り時々晴れ（深夜0時台に小雨の可能性がありますが、日中は雨の心配はほとんどありません）
  - **降水確率**: 10% 〜 37%（日中は概ね10〜30%前後）
  - **雨量**: 0.0 mm
  - **状況**: 日中の移動や観光で傘を使う場面はほとんどなさそうです。

- **2026年9月6日（日・2日目）**
  - **天候**: 朝から夜にかけて**雨**（weather_code: 51〜63）
  - **降水確率**: 午前中から上昇し、午後には **70% 〜 78%**
  - **雨量**: 0.5 〜 3.0 mm/h 程度のまとまった降雨が続く見込み
  - **状況**: 外出時には**傘（折りたたみ傘または雨傘）が必須**となります。

- **2026年9月7日（月・3日目）**
  - **天候**: **雷雨および雨**（weather_code: 95の雷雨予報あり）
  - **降水確率**: 早朝から昼過ぎにかけて **60% 〜 84%**
  - **雨量**: 未明から朝方を中心に雨が降り、雷雨の可能性があります
  - **状況**: 午前中を中心に激しい雨や雷の恐れがあるため、**しっかりとした雨具の携行**をおすすめします。

---

### まとめ
旅行初日の9月5日は天気が持ちそうですが、**9月6日・9月7日は降水確率が高く雨・雷雨が予想されています**ので、折りたたみ傘や雨具を旅行カバンに入れておくことをおすすめします。

**[TMF-10]**

AIエージェントからの応答イベントを確認します。ファンクションコールで `get_weather_forcast` が使用されたことがわかります。

In [93]:
for event in events:
    print('====')
    print(event['content'])

====
{'parts': [{'function_call': {'id': 'call_235021', 'args': {'prefecture_name': '京都府'}, 'name': 'get_weather_forecast'}, 'thought_signature': 'AY89a19hjxUSb-_05RfacJw6pMRUNPoxY6JEvr7QL1ZCOBAsV4NBA4H74MPr-gXOggpDOT44ieLM_9xYaNYW03k0b7ystrp15zzN_80E4SJzQ3EIEzT4FDabn7gRb9gEFtuoKePtrWH3xRNFoxWdejghU-zju1ByX120sHfB_-qWu-92limZZ39Xd-E2li5Jv5ZVGqyYv3FsKd8eY_kxAsSjY6eElY01PoJFW2UgpoUKZ1gru4kcGay0h54FWKvUchKaZPh3aJ1gKTjonNwBugt-qa4D5HXDgBvgj6UM1FXsTZRaY89yzfGldijmKByiZlKD1shTdTekKw3I'}], 'role': 'model'}
====
{'parts': [{'function_response': {'id': 'call_235021', 'name': 'get_weather_forecast', 'response': {'latitude': 35.0, 'longitude': 135.75, 'generationtime_ms': 1.7180442810058594, 'utc_offset_seconds': 32400, 'timezone': 'Asia/Tokyo', 'timezone_abbreviation': 'GMT+9', 'elevation': 56.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation_probability': '%', 'rain': 'mm', 'weather_code': 'wmo code'}, 'hourly': {'time': ['2026-09-04T00:00', '2026-09-04T01:00', '2026

**[TMF-11]**

関数 `get_weather_forcast` の docstring に記載した「当日から始まる7日間（168時間）の天気予報」という内容を理解しているか確認します。

In [89]:
query = '''
なぜ明日が2026年9月5日とわかったのですか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

天気予報ツール（API）から取得した気象データそのものに日時が含まれていたためです。

ツールを実行して取得したデータの開始日時が `2026-09-04T00:00`（2026年9月4日 0時）から始まっており、APIの仕様が「**今日から始まる7日間の天気予報**」を取得する仕組みとなっているため、

* **今日**：2026年9月4日
* **明日（旅行1日目）**：2026年9月5日

と判断してご案内いたしました。

## MCP サーバーを利用する例

事前準備として、クラウドコンソールの「Google Maps Platform」→「鍵と認証情報」で API キーを取得して、Colaboratory のシークレットに `google_maps_api_key` という名前で保存しておきます。

**[TMF-12]**

Google Maps Grounding Lite MCP の公開 MCP サーバーを利用するツールセットを定義します。

In [95]:
google_maps_toolset = McpToolset(
        connection_params=StreamableHTTPServerParams(
            # Google Maps Grounding Lite MCP エンドポイント
            url='https://mapstools.googleapis.com/mcp',
            headers={'X-Goog-Api-Key': userdata.get('google_maps_api_key')},
        ),
        tool_filter=['search_places', 'compute_routes'] # 'lookup_weather'
    )

**[TMF-13]**

定義したツールセットをツール関数に持つ LlmAgent オブジェクトと AdkApp オブジェクトを作成します。

In [98]:
instruction='''
あなたは Google Maps を活用して施設情報を提供するエージェントです。
- Google Maps ツールを使用して得られた最新情報に基づいて回答してください。
- 可能な場合は Google Maps リンクを提供してください。
'''

facility_information_agent = LlmAgent(
    name='facility_information_agent',
    model='gemini-3.8-flash',
    description='Google Maps に基づいて施設情報を提供するエージェント',
    instruction=instruction,
    tools=[google_maps_toolset],
    planner=BuiltInPlanner(
        thinking_config=ThinkingConfig(
            thinking_level=ThinkingLevel.LOW,
            include_thoughts=False,
        ),
    ),
)

facility_information_app = AdkApp(
    agent=facility_information_agent,
    app_name='facility_information_app',
)

**[TMF-14]**

作成したAIエージェントと会話します。MCP サーバーから取得したツールの機能を理解していることを確認します。

In [113]:
chat_client = ChatClient(facility_information_app)

query = '''
何ができますか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

私はGoogle Mapsの情報を活用して、施設や場所に関する様々な情報を提供するエージェントです。主に以下のようなお手伝いができます。

1. **施設の検索・詳細情報の確認**
   - 特定の店舗、レストラン、観光スポット、公共施設などの検索
   - 住所、営業時間、電話番号などの基本情報の確認
   - 写真や口コミ、Google Mapsのリンクの案内

2. **条件に合わせたスポット探し**
   - 「渋谷駅近くの落ち着いたカフェ」
   - 「新宿でテラス席のあるイタリアン」
   - 「子供連れで行きやすい公園」などの条件検索

3. **ルート案内・所要時間の確認**
   - 出発地から目的地までのルート検索（車または徒歩）
   - 移動距離や所要時間の目安の案内

「〇〇周辺のおすすめのランチを知りたい」「東京タワーの営業時間やアクセスを教えてほしい」など、気になる場所や行きたいスポットがあれば、お気軽にご質問ください！

**[TMF-15]**

具体的な施設の検索を行ってみます。

In [115]:
query = '''
渋谷駅の近くにある郵便局を教えて。特にハチ公前出口から最も近いのはどれですか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

渋谷駅周辺にはいくつか郵便局がありますが、**ハチ公前出口（広場）から最も近い**のは **[渋谷中央街郵便局](https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188b57b27769c5:0xd42316e784b9ef7)** です。

---

### 1. [渋谷中央街郵便局](https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188b57b27769c5:0xd42316e784b9ef7)（ハチ公前から最も近い）
* **ハチ公前からの距離・時間**: 徒歩約5〜6分（約410m）
* **住所**: 東京都渋谷区道玄坂1丁目10-2 渋谷Crビル 1F
* **営業時間**: 月曜〜金曜 9:00〜17:00（土日祝は休業）
* **特徴**: ハチ公前広場からスクランブル交差点を渡り、渋谷フクラスや道玄坂方面へ進んだ路地沿いにあります。

---

### その他の近い郵便局

* **[渋谷神南郵便局](https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188ca8882f21bb:0xe41fdf24841341b4)**
  * ハチ公前から徒歩約6分（約440m）
  * 住所: 東京都渋谷区神南1丁目21-1 日本生命渋谷ビル
  * 営業時間: 平日 9:00〜17:00（土日祝休）
  * モディやタワーレコード方面（北側）へ向かう場合はこちらが便利です。

* **[渋谷郵便局](https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188b588bd4c71b:0x4ebc5f0fa6782e4e)**（地域の中核本局）
  * ハチ公前から徒歩約8分（宮益坂方面）
  * 住所: 東京都渋谷区渋谷1丁目12-13
  * 営業時間: 平日 9:00〜21:00、土曜・日曜・祝日 9:00〜18:00
  * **平日夕方以降や土日・祝日に利用したい場合**（ゆうゆう窓口含む）はこちらがおすすめです。

※徒歩ルートの所要時間は歩行速度や混雑具合により前後する場合があります。

**[TMF-16]**

AIエージェントからの応答イベントを確認します。この例では、ファンクションコールが複数回実行されており、`search_places` と `compute_routes` のツールが使用されています。

In [117]:
for event in events:
    print('====')
    print(event['content'])

====
{'parts': [{'function_call': {'id': 'call_252209', 'args': {'languageCode': 'ja', 'textQuery': '渋谷駅 ハチ公前 郵便局'}, 'name': 'search_places'}, 'thought_signature': 'AY89a19jx6yuk7AiBsJ5373Awzzhawn6hSmdc-EcXQPjNXXLYaR3-eN_t0muzeZ2t9a0J74zreiYu37OYKdXS7G7X5Ab_kSLgYzMspUY'}], 'role': 'model'}
====
{'parts': [{'function_response': {'id': 'call_252209', 'name': 'search_places', 'response': {'content': [{'type': 'text', 'text': '{\n  "places": [\n    {\n      "place": "places/ChIJG8fUi1iLGGARTi54pg9fvE4",\n      "id": "ChIJG8fUi1iLGGARTi54pg9fvE4",\n      "location": {\n        "latitude": 35.6599551,\n        "longitude": 139.7041647\n      },\n      "googleMapsLinks": {\n        "directionsUrl": "https://www.google.com/maps/dir//\'\'/data=!4m7!4m6!1m1!4e2!1m2!1m1!1s0x60188b588bd4c71b:0x4ebc5f0fa6782e4e!3e0",\n        "placeUrl": "https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188b588bd4c71b:0x4ebc5f0fa6782e4e",\n        "writeAReviewUrl": "https://www.google.com/maps/place//data=!4

**[TMF-16]**

どのようにして最も近い郵便局を検索したのか確認します。

In [118]:
query = '''
なぜ渋谷中央街郵便局が最も近いとわかったの？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

「渋谷中央街郵便局」が最も近いと判断した理由は、**Google Maps の経路計算（ルート検索）機能を用いて、ハチ公前広場からの実際の徒歩移動距離と所要時間を算出して比較したため**です。

具体的には以下のような手順で確認しました：

1. **周辺の郵便局の位置を特定**
   渋谷駅周辺の郵便局を検索し、ハチ公前出口と同じ西口・北口側にある主要な候補（道玄坂側の「渋谷中央街郵便局」と神南側の「渋谷神南郵便局」など）を抽出しました。

2. **ハチ公前広場からの徒歩ルートを計算**
   - **渋谷中央街郵便局**: 徒歩ルートの距離 **約413m**（所要時間：約5分58秒）
   - **渋谷神南郵便局**: 徒歩ルートの距離 **約437m**（所要時間：約6分21秒）
   - **渋谷郵便局（宮益坂）**: 駅の反対側（東口・宮益坂側）にあり、線路下や地下通路を抜ける必要があるため徒歩約8分以上

このように直線距離だけでなく、道路網に基づいた**実際の歩行距離（約413m）が最も短かった**ことから、ハチ公前出口から一番近い郵便局としてご案内しました。

In [119]:
print(response)

「渋谷中央街郵便局」が最も近いと判断した理由は、**Google Maps の経路計算（ルート検索）機能を用いて、ハチ公前広場からの実際の徒歩移動距離と所要時間を算出して比較したため**です。

具体的には以下のような手順で確認しました：

1. **周辺の郵便局の位置を特定**
   渋谷駅周辺の郵便局を検索し、ハチ公前出口と同じ西口・北口側にある主要な候補（道玄坂側の「渋谷中央街郵便局」と神南側の「渋谷神南郵便局」など）を抽出しました。

2. **ハチ公前広場からの徒歩ルートを計算**
   - **渋谷中央街郵便局**: 徒歩ルートの距離 **約413m**（所要時間：約5分58秒）
   - **渋谷神南郵便局**: 徒歩ルートの距離 **約437m**（所要時間：約6分21秒）
   - **渋谷郵便局（宮益坂）**: 駅の反対側（東口・宮益坂側）にあり、線路下や地下通路を抜ける必要があるため徒歩約8分以上

このように直線距離だけでなく、道路網に基づいた**実際の歩行距離（約413m）が最も短かった**ことから、ハチ公前出口から一番近い郵便局としてご案内しました。
